In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
def get_stock_data(ticker, period="1y"):
    stock = yf.download(ticker, period=period, multi_level_index= False)
    return stock
    """
    Download historical stock data.
    ticker = stock symbol e.g "AAPL"
    period = how far back e.g "1y", "6mo", "2y"
    """
if __name__ =="__main__":
    data = get_stock_data("AAPL")
    print(f"\nFirst 5 rows:\n {data.head(5)}")
    print(f"\nLast 5 rows:\n {data.tail(5)})")
    print("\nShape:", data.shape)
    print("\nColumns:", data.columns.tolist())


In [ ]:
Close = data["Close"].squeeze() #Extract closing price
data["Daily_return"] = Close.pct_change().fillna(0) #Daily returnn
data["Cumulative_return"] = (1 + data["Daily_return"]).cumprod() #Cumulative return
# Moving Averages
data["MA_50"] = Close.rolling(window=50).mean()
data["MA_200"] = Close.rolling(window=200).mean()

#Volatility (annualized)
daily_vol = data["Daily_return"].std()
annual_vol = daily_vol * np.sqrt(252)

# Sharpe ratio
avg_daily_return = data["Daily_return"].mean()
sharpe_ratio = (avg_daily_return / daily_vol) * np.sqrt(252)

print(f"Financial summary for AAPL\n")
print(f"Start Price: ${float(Close.iloc[0]):.2f}")
print(f"End Price: ${float(Close.iloc[-1]):.2f}")
print(f"Cumulative_return: {float(data['Cumulative_return'].iloc[-1]) * 100:.2f}%")
print(f"Annualized vol: {float(annual_vol) * 100:.2f}%")
print(f"Sharpe ratio: {float(sharpe_ratio):.2f}")


In [ ]:
# Chart 1 - Price with Moving Averages
plt.figure(figsize=(14,5))
plt.plot(data["Close"], label="Close Price", color="blue")
plt.plot(data["MA_50"], label="50-Day MA", color="orange")
plt.plot(data["MA_200"], label="200-Day MA", color="red")
plt.title("AAPL Stock Price with Moving Averages")
plt.ylabel("Price (USD)")
plt.legend()
plt.grid(True)
plt.show()

# Chart 2 - Daily returns
plt.figure(figsize=(14, 4))
plt.plot(data["Daily_return"], color="green")
plt.axhline(y=0, color="black", linestyle="--")
plt.title("Daily returns")
plt.grid(True)
plt.show()

# Chart 3 - Cumulative returns
plt.figure(figsize=(14,4))
plt.plot(data["Cumulative_return"], color="purple")
plt.axhline(y=1, color="black", linestyle="--")
plt.title("Cumulative Retuens")
plt.grid(True)
plt.show()


In [ ]:
# Chart 1 - Distribution of Daily Returns
plt.figure(figsize=(10,5))
sns.histplot(data["Daily_return"].dropna(), bins=50, kde=True, color="blue")
plt.title("Distribution of daily Return - AAPL")
plt.xlabel("Daily Return")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

# Chart 2 - Correlation Heatmap (Multiple Stocks)
tickers = ["AAPL", "TSLA", "MSFT", "GOOGL", "AMZN"]
multi_data = yf.download(tickers, period="1y")["Close"]
returns_all = multi_data.pct_change().dropna()
correlation = returns_all.corr()

plt.figure(figsize=(8,6))
sns.heatmap(correlation, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("stock correlation heatmap")
plt.show()


In [ ]:
# final summary cell
print("=" * 45)
print("     STOCK DASHBOARD - FINAL REPORT   ")
print("=" * 45)
print(f"\nStock:AAPL")
print(f"\nPeriod:1 Year")
print(f"\nStart Price:${float(Close.iloc[0]):.2f}")
print(f"\nEnd Price:${float(Close.iloc[-1]):.2f}")
print(f"\nTotal Return:{(float(data['Cumulative_return'].iloc[-1])-1)* 100:.2f}%")
print(f"\nVolatity:{float(annual_vol)* 100:.2f}%")
print(f"\nsharp ratio:{float(sharpe_ratio):2f}")
print(f"\n"+"="*45)

# Sharpe ratio verdict
if sharpe_ratio>2:
    print("verdict: excellent risk-adjusted return!")
elif sharpe_ratio>1:
    print("verdict: good risk-adjusted return!")
else:
    print("verdict: Poor risk-adjusted return!")

print("="*45)
